# Task 11 — XLM-R-base Reference Experiments

**Protocol:** `phase2-v1` | **Backbone:** `xlm-roberta-base` | **Seeds:** `0, 1, 2`

Execution order (respecting dependencies):
1. **Smoke test** (1 run, dev-only) — verify GPU & BF16 pipeline
2. **DFT LR sweep** (6 runs, dev-only, `seed 0`) — lock DFT LR for `SA` and `SB`
3. **DFT 100%** (6 runs, `seed 0..2`) — 100% reference + source checkpoints for `Seq` & `ZS`
4. **Seq LR sweep** (6 runs, dev-only, `seed 0`) — uses `DFT 100%` checkpoints, then locks `selection_xlmr_lr.json`
5. **DFT 10%** (6 runs, `seed 0..2`) — target low-resource baseline
6. **Seq 10%** (6 runs, `seed 0..2`) — sequential transfer from `DFT 100%` source checkpoints
7. **ZS inference** (6 runs, `seed 0..2`) — zero-shot inference (`100% source -> test target`, no training)
8. **Results & Cost Measurement** — VRAM, latency (`batch=1` & `batch=32`), throughput, GPU-hours/1000 samples

> ⚠️ Switch Colab runtime to **GPU (A100 preferred)** before running.

## 0. Setup

In [ ]:
import os, subprocess, sys

REPO_URL = 'https://github.com/PhuongNgoVan/Xdomain-Transfer-Learning.git'
BRANCH = 'phase2_task11-xlmr-llm'
WORK_DIR = '/content/Xdomain-Transfer-Learning'
RESULTS_DIR = 'results/phase2'
CKPT_DIR = 'checkpoints/phase2'

if not os.path.isdir(WORK_DIR):
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch',
                    '--no-checkout', REPO_URL, WORK_DIR], check=True)
os.chdir(WORK_DIR)
subprocess.run(['git', 'checkout', BRANCH], check=True)
subprocess.run(['git', 'pull', '--ff-only'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('Setup complete.')

In [ ]:
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'BF16 supported: {torch.cuda.is_bf16_supported()}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

## 1. Smoke Test (dev-only)

In [ ]:
!python -X utf8 -m src.phase2.cli \
    --config configs/task11_xlmr_smoke.yaml \
    --max-examples 50

## 2. DFT LR Sweep (6 dev-only runs, `seed 0`)

In [ ]:
import subprocess, sys
import pandas as pd
from pathlib import Path

for direction in ('SA', 'SB'):
    for lr in (2e-5, 3e-5, 5e-5):
        cmd = [sys.executable, '-X', 'utf8', '-m', 'src.phase2.cli',
               '--config', 'configs/task11_xlmr_sweep.yaml',
               '--direction', direction, '--method', 'dft', '--lr', str(lr),
               '--results-dir', RESULTS_DIR]
        print(f'\n>>> DFT sweep {direction} lr={lr}')
        subprocess.run(cmd, check=True)

runs = pd.read_csv(Path(RESULTS_DIR) / 'phase2_runs.csv', keep_default_na=False)
dft_lr = {}
for direction in ('SA', 'SB'):
    view = 'V3-NEU' if direction == 'SA' else 'V5-VSMEC'
    block = runs[(runs.backbone == 'XLMR') & (runs.run_mode == 'sweep') &
                 (runs.method == 'dft') & (runs.direction == direction) &
                 (runs.label_view == view) & (runs.map_method == 'none')]
    best = block.sort_values(['dev_selection_score', 'lr'], ascending=[False, True]).iloc[0]
    dft_lr[direction] = float(best.lr)
    print(f'Selected DFT LR for {direction}: {dft_lr[direction]} (dev={best.dev_selection_score:.4f})')

## 3. DFT 100% — Reference & Source Checkpoints for Seq/ZS (6 runs)

In [ ]:
import subprocess, sys

for direction in ('SA', 'SB'):
    lr = dft_lr[direction]
    for seed in (0, 1, 2):
        cmd = [sys.executable, '-X', 'utf8', '-m', 'src.phase2.cli',
               '--config', 'configs/phase2.yaml',
               '--backbone', 'XLMR', '--direction', direction,
               '--method', 'dft', '--fraction', '100',
               '--seed', str(seed), '--lr', str(lr),
               '--run-mode', 'main',
               '--results-dir', RESULTS_DIR,
               '--ckpt-out-dir', CKPT_DIR]
        print(f'\n>>> DFT 100% {direction} seed={seed} lr={lr}')
        subprocess.run(cmd, check=True)
print('\nDFT 100% complete.')

## 4. Seq LR Sweep (6 dev-only runs, `seed 0`) & Lock `selection_xlmr_lr.json`

In [ ]:
import json, subprocess, sys
from pathlib import Path

for direction in ('SA', 'SB'):
    for lr in (1e-5, 2e-5, 3e-5):
        cmd = [sys.executable, '-X', 'utf8', '-m', 'src.phase2.cli',
               '--config', 'configs/task11_xlmr_sweep.yaml',
               '--direction', direction, '--method', 'seq', '--lr', str(lr),
               '--results-dir', RESULTS_DIR,
               '--source-ckpt-root', CKPT_DIR]
        print(f'\n>>> Seq sweep {direction} lr={lr}')
        subprocess.run(cmd, check=True)

subprocess.run([sys.executable, '-X', 'utf8', '-m', 'src.phase2.select_lr',
                '--results-dir', RESULTS_DIR, '--backbone', 'XLMR'], check=True)

sel = json.loads((Path(RESULTS_DIR) / 'selection_xlmr_lr.json').read_text(encoding='utf-8'))
for key, val in sel['selected'].items():
    print(f"  {key}: LR={val['lr']}, dev_score={val['dev_selection_score']:.4f}, edge={val['grid_edge']}")

## 5. DFT 10% — Target Low-Resource Baseline (6 runs)

In [ ]:
import json, subprocess, sys
from pathlib import Path

sel_path = str(Path(RESULTS_DIR) / 'selection_xlmr_lr.json')
sel = json.loads(Path(sel_path).read_text(encoding='utf-8'))

for direction in ('SA', 'SB'):
    lr = sel['selected'][f'XLMR/{direction}/dft']['lr']
    for seed in (0, 1, 2):
        cmd = [sys.executable, '-X', 'utf8', '-m', 'src.phase2.cli',
               '--config', 'configs/phase2.yaml',
               '--backbone', 'XLMR', '--direction', direction,
               '--method', 'dft', '--fraction', '10',
               '--seed', str(seed), '--lr', str(lr),
               '--run-mode', 'main',
               '--results-dir', RESULTS_DIR,
               '--ckpt-out-dir', CKPT_DIR,
               '--selection-path', sel_path]
        print(f'\n>>> DFT 10% {direction} seed={seed} lr={lr}')
        subprocess.run(cmd, check=True)
print('\nDFT 10% complete.')

## 6. Seq 10% — Sequential Transfer (6 runs)

In [ ]:
import json, subprocess, sys
from pathlib import Path

sel_path = str(Path(RESULTS_DIR) / 'selection_xlmr_lr.json')
sel = json.loads(Path(sel_path).read_text(encoding='utf-8'))

for direction in ('SA', 'SB'):
    lr = sel['selected'][f'XLMR/{direction}/seq']['lr']
    for seed in (0, 1, 2):
        cmd = [sys.executable, '-X', 'utf8', '-m', 'src.phase2.cli',
               '--config', 'configs/phase2.yaml',
               '--backbone', 'XLMR', '--direction', direction,
               '--method', 'seq', '--fraction', '10',
               '--seed', str(seed), '--lr', str(lr),
               '--run-mode', 'main',
               '--results-dir', RESULTS_DIR,
               '--ckpt-out-dir', CKPT_DIR,
               '--source-ckpt-root', CKPT_DIR,
               '--selection-path', sel_path]
        print(f'\n>>> Seq 10% {direction} seed={seed} lr={lr}')
        subprocess.run(cmd, check=True)
print('\nSeq 10% complete.')

## 7. Zero-Shot (ZS) Inference — Two-way (`seed 0..2`, no training)

In [ ]:
import numpy as np
import pandas as pd
import torch
from pathlib import Path

from src.data.preprocess import preprocess_text
from src.phase2.models import _locate_phase2_xlmr_source_checkpoint
from src.plm.model_ckpt import Stage1Classifier
from src.zs.infer import _atomic_csv, _prediction_frame, score_prediction_view
from src.zs.mapping import validate_probabilities
from src.zs.views import load_test_frame, view_jobs

zs_rows = []
zs_out_dir = Path(RESULTS_DIR) / 'xlmr_zs'
zs_out_dir.mkdir(parents=True, exist_ok=True)

for direction in ('SA', 'SB'):
    job = 'zsA' if direction == 'SA' else 'zsB'
    target_domain = 'neuesc' if direction == 'SA' else 'vsmec'
    src_labels = 7 if direction == 'SA' else 3
    test_df = load_test_frame('data/clean', target_domain)
    texts = [preprocess_text(t, emoji_mode='keep') for t in test_df['text'].astype(str)]
    for seed in (0, 1, 2):
        ckpt = _locate_phase2_xlmr_source_checkpoint(CKPT_DIR, direction, seed)
        model, tokenizer = Stage1Classifier.load_stage1_checkpoint(ckpt, device='cuda')
        model.float().eval()
        probs_list = []
        with torch.inference_mode():
            for i in range(0, len(texts), 32):
                enc = tokenizer(texts[i:i+32], padding=True, truncation=True,
                                max_length=256, return_tensors='pt').to('cuda')
                logits = model(**enc).logits
                probs_list.append(logits.float().softmax(-1).cpu().numpy())
        probs = validate_probabilities(np.vstack(probs_list), src_labels)
        pred_frame = _prediction_frame(test_df, target_domain, src_labels, probs)
        pred_path = zs_out_dir / f'XLMR_{direction}_zs_s{seed}_predictions_test.csv'
        _atomic_csv(pred_frame, pred_path)
        for view, map_method in view_jobs(job):
            metrics = score_prediction_view(pd.read_csv(pred_path), view, map_method)
            zs_rows.append({'backbone': 'XLMR', 'direction': direction, 'method': 'zs',
                            'seed': seed, 'label_view': view, 'map_method': map_method, **metrics})

zs_summary = pd.DataFrame(zs_rows)
_atomic_csv(zs_summary, zs_out_dir / 'xlmr_zs_summary.csv')
print(zs_summary[['direction', 'seed', 'label_view', 'map_method', 'macro_f1']])

## 8. Results & Cost Measurement

In [ ]:
import pandas as pd
from pathlib import Path

runs = pd.read_csv(Path(RESULTS_DIR) / 'phase2_runs.csv', keep_default_na=False)
xlmr = runs[(runs.backbone == 'XLMR') & (runs.run_mode == 'main')].copy()
for direction in ('SA', 'SB'):
    view = 'V3-NEU' if direction == 'SA' else 'V7-VSMEC'
    sub = xlmr[(xlmr.direction == direction) & (xlmr.split == 'test') & (xlmr.label_view == view)]
    print(f'=== {direction} ({view}) ===')
    for _, row in sub.iterrows():
        print(f'  {row.method} f{row.fraction}% s{row.seed}: macro_f1={row.macro_f1:.4f}')

In [ ]:
import json, glob, time, torch
import numpy as np
from pathlib import Path
from transformers import AutoModel, AutoTokenizer
from src.common.seed import set_seed
from src.plm.model_ckpt import Stage1Classifier
from src.phase2.data import load_phase2_prepared_data
from src.phase2.protocol import load_phase2_config

for cost_file in sorted(glob.glob(f'{RESULTS_DIR}/runs/XLMR_*/cost.json')):
    cost = json.loads(Path(cost_file).read_text(encoding='utf-8'))
    print(Path(cost_file).parent.name, cost.get('total_params'), cost.get('peak_vram_mb'), cost.get('train_seconds'))

cfg = load_phase2_config('configs/phase2.yaml')
test_texts = load_phase2_prepared_data('SA', 10, 0, 'XLMR', cfg, run_mode='main')['test_texts']
encoder = AutoModel.from_pretrained('xlm-roberta-base')
tokenizer = AutoTokenizer.from_pretrained('xlm-roberta-base')
set_seed(0)
model = Stage1Classifier(encoder, 3, dropout=0.1).cuda().eval()

for _ in range(3):
    enc = tokenizer(test_texts[:1], return_tensors='pt', padding=True, truncation=True, max_length=256)
    with torch.inference_mode():
        model(**{k: v.cuda() for k, v in enc.items()})
torch.cuda.synchronize()

times_b1 = []
for text in test_texts[:100]:
    enc = tokenizer([text], return_tensors='pt', padding=True, truncation=True, max_length=256)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.inference_mode():
        model(**{k: v.cuda() for k, v in enc.items()})
    torch.cuda.synchronize()
    times_b1.append(time.perf_counter() - t0)

torch.cuda.synchronize()
t0 = time.perf_counter()
for i in range(0, len(test_texts), 32):
    enc = tokenizer(test_texts[i:i+32], return_tensors='pt', padding=True, truncation=True, max_length=256)
    with torch.inference_mode():
        model(**{k: v.cuda() for k, v in enc.items()})
torch.cuda.synchronize()
total_time = time.perf_counter() - t0
n = len(test_texts)
print(f'Latency batch=1 mean: {np.mean(times_b1)*1000:.2f} ms | p95: {np.percentile(times_b1, 95)*1000:.2f} ms')
print(f'Throughput batch=32: {n/total_time:.1f} samples/s | GPU-h/1000: {total_time/n/3.6:.4f}')

## 9. Archive & Download Results

In [ ]:
!cd {WORK_DIR} && tar czf /content/task11_xlmr_results.tar.gz \
    results/phase2/phase2_runs.csv \
    results/phase2/selection_xlmr_lr.json \
    results/phase2/xlmr_lr_selection.csv \
    results/phase2/runs/XLMR_*/ \
    results/phase2/xlmr_zs/

from google.colab import files
files.download('/content/task11_xlmr_results.tar.gz')